# Alarm Cluster Visualizations

For each alarm cluster, generate an interactive HTML plot showing:
1. **Main plot**: All PV and OP tags (min-max normalized, original values on hover)
2. **Bottom subplot**: Control actions timeline (SP/OP: red=decrease, green=increase; MODE: blue; others: grey)
3. **Shading**: Cluster period (light orange) + individual alarm periods (light red)

Time window: `[cluster_start - 30 min, cluster_end + 30 min]`

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path

In [2]:
# Load data
DATA_DIR = Path('../DATA')
RESULTS_DIR = Path('../RESULTS/cluster_visualizations')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load PV/OP time series
op_pv_data_df = pd.read_parquet(DATA_DIR / '03LIC_1071_JAN_2026_filtered.parquet')
op_pv_data_df.sort_index(inplace=True)

# Load alarm clusters and control actions
alarms_df = pd.read_excel(DATA_DIR / '1071_pvlo_alarms_clustered_with_control_actions.xlsx', sheet_name=0)
actions_df = pd.read_excel(DATA_DIR / '1071_pvlo_alarms_clustered_with_control_actions.xlsx', sheet_name=1)

# Load raw deduplicated events (for custom window plots)
raw_events_df = pd.read_csv(DATA_DIR / 'trip_filtered_events_dedup.csv', low_memory=False)
raw_events_df['VT_Start'] = pd.to_datetime(raw_events_df['VT_Start'])
raw_events_df = raw_events_df.sort_values('VT_Start').reset_index(drop=True)
# Pre-filter to CHANGE events only
raw_change_events = raw_events_df[raw_events_df['ConditionName'] == 'CHANGE'].copy()

# Load operating limits for 03LIC_1071
operating_limits_df = pd.read_csv(DATA_DIR / 'operating_limits.csv')
target_pv_limits = operating_limits_df[operating_limits_df['TAG_NAME'] == '03LIC_1071.PV'].iloc[0]

TARGET_PV_LOWER = target_pv_limits['LOWER_LIMIT']  # 35.25
TARGET_PV_UPPER = target_pv_limits['UPPER_LIMIT']  # 42.41
TARGET_ALARM_LIMIT = 28.75

print(f'PV/OP data: {op_pv_data_df.shape[0]:,} rows, {op_pv_data_df.index.min()} to {op_pv_data_df.index.max()}')
print(f'Alarm clusters: {alarms_df["cluster_id"].nunique()} clusters, {len(alarms_df)} individual alarms')
print(f'Control actions (Excel): {len(actions_df)} actions across {actions_df["cluster_id"].nunique()} clusters')
print(f'Raw CHANGE events (dedup): {len(raw_change_events):,}')
print(f'03LIC_1071.PV limits: lower={TARGET_PV_LOWER:.2f}, upper={TARGET_PV_UPPER:.2f}, alarm={TARGET_ALARM_LIMIT}')

PV/OP data: 1,718,039 rows, 2022-01-03 22:45:00 to 2025-06-23 20:44:00
Alarm clusters: 539 clusters, 1379 individual alarms
Control actions (Excel): 16094 actions across 430 clusters
Raw CHANGE events (dedup): 115,857
03LIC_1071.PV limits: lower=35.25, upper=42.41, alarm=28.75


In [3]:
# Identify tag columns and group by base tag name
tag_cols = [c for c in op_pv_data_df.columns if c.endswith('.PV') or c.endswith('.OP')]
pv_cols = sorted([c for c in tag_cols if c.endswith('.PV')])
op_cols = sorted([c for c in tag_cols if c.endswith('.OP')])

# Build ordered list: for each base tag, PV first then OP
pv_bases = {c.replace('.PV', ''): c for c in pv_cols}
op_bases = {c.replace('.OP', ''): c for c in op_cols}
all_bases = sorted(set(list(pv_bases.keys()) + list(op_bases.keys())))

# Build ordered tag list grouped by base name
ordered_tags = []
for base in all_bases:
    if base in pv_bases:
        ordered_tags.append(pv_bases[base])
    if base in op_bases:
        ordered_tags.append(op_bases[base])

print(f'Total tags to plot: {len(ordered_tags)} ({len(pv_cols)} PV + {len(op_cols)} OP)')
print(f'Base tags: {len(all_bases)}')

Total tags to plot: 43 (28 PV + 15 OP)
Base tags: 28


In [4]:
# Build cluster info: one row per cluster with start, end, and list of individual alarms
cluster_info = {}
for cid, grp in alarms_df.groupby('cluster_id'):
    cluster_info[cid] = {
        'cluster_start': grp['cluster_start_time'].iloc[0],
        'cluster_end': grp['cluster_end_time'].iloc[0],
        'cluster_type': grp['cluster_type'].iloc[0],
        'total_alarms': grp['cluster_total_alarms'].iloc[0],
        'alarms': list(grp[['alarm_start', 'alarm_end', 'episode_num']].itertuples(index=False, name=None)),
    }

# Build actions per cluster
cluster_actions = {}
for cid, grp in actions_df.groupby('cluster_id'):
    cluster_actions[cid] = grp.copy()

print(f'Total clusters: {len(cluster_info)}')
print(f'Clusters with actions: {len(cluster_actions)}')

Total clusters: 539
Clusters with actions: 430


In [5]:
# Generate a consistent color palette for tags using plotly qualitative colors
import plotly.express as px

# Use a large qualitative palette
palette = (
    px.colors.qualitative.Dark24 +
    px.colors.qualitative.Light24 +
    px.colors.qualitative.Alphabet
)

# Assign colors by base tag (PV and OP of same base share same color)
base_colors = {}
for i, base in enumerate(all_bases):
    base_colors[base] = palette[i % len(palette)]

print(f'Color assignments for {len(base_colors)} base tags')

Color assignments for 28 base tags


In [6]:
def create_window_plot(window_start, window_end, op_pv_df, ordered_tags, base_colors,
                       title='', actions=None, cluster_regions=None, alarm_regions=None,
                       target_limits=None):
    """
    General-purpose plot for any time window.

    Parameters:
        window_start, window_end: pd.Timestamp - time window boundaries
        op_pv_df: DataFrame with TimeStamp index and PV/OP columns
        ordered_tags: list of column names to plot
        base_colors: dict mapping base tag name to color
        title: str - plot title
        actions: DataFrame of control actions (needs Source, Description, action_direction,
                 VT_Start, PrevValue, Value, action_timing columns). None if no actions.
        cluster_regions: list of (cluster_id, start, end) tuples for orange shading
        alarm_regions: list of (alarm_start, alarm_end, episode_num) tuples for red shading
        target_limits: dict with 'lower', 'upper', 'alarm' for 03LIC_1071.PV horizontal lines
    """
    # Extract PV/OP data for window
    mask = (op_pv_df.index >= window_start) & (op_pv_df.index <= window_end)
    window_df = op_pv_df.loc[mask, [c for c in ordered_tags if c in op_pv_df.columns]].copy()

    if window_df.empty:
        print(f'  No PV/OP data in window {window_start} to {window_end}')
        return None

    has_actions = actions is not None and len(actions) > 0

    # Determine subplot heights
    if has_actions:
        n_action_tags = actions['Source'].nunique()
        action_height = max(0.15, min(0.35, n_action_tags * 0.04))
        row_heights = [1 - action_height, action_height]
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                            row_heights=row_heights,
                            subplot_titles=('PV / OP Trends (normalized)', 'Control Actions'))
    else:
        fig = make_subplots(rows=1, cols=1,
                            subplot_titles=('PV / OP Trends (normalized)',))

    # ── Top subplot: PV/OP traces (min-max normalized) ──
    for col in ordered_tags:
        if col not in window_df.columns:
            continue
        series = window_df[col]
        if series.isna().all():
            continue

        col_min, col_max = series.min(), series.max()
        if col_max == col_min:
            normalized = pd.Series(0.5, index=series.index)
        else:
            normalized = (series - col_min) / (col_max - col_min)

        base_tag = col.rsplit('.', 1)[0]
        suffix = col.rsplit('.', 1)[1]
        color = base_colors.get(base_tag, '#888888')
        is_op = suffix == 'OP'
        is_target = base_tag == '03LIC_1071'

        fig.add_trace(
            go.Scatter(
                x=series.index, y=normalized, mode='lines', name=col,
                legendgroup=base_tag,
                legendgrouptitle_text=base_tag if suffix == 'PV' else None,
                line=dict(color=color, width=2.5 if is_target else 1.5,
                          dash='dot' if is_op else 'solid'),
                visible=True if is_target else 'legendonly',
                customdata=np.column_stack([series.values]),
                hovertemplate=(f'<b>{col}</b><br>Time: %{{x}}<br>'
                               'Value: %{customdata[0]:.4f}<br><extra></extra>'),
            ),
            row=1, col=1
        )

    # ── Horizontal limit lines for 03LIC_1071.PV ──
    if target_limits and '03LIC_1071.PV' in window_df.columns:
        pv_series = window_df['03LIC_1071.PV']
        pv_min, pv_max = pv_series.min(), pv_series.max()
        if pv_max != pv_min:
            for limit_val, limit_name, limit_color, limit_dash in [
                (target_limits['alarm'], f'({target_limits["alarm"]})', 'red', 'dash'),
                (target_limits['lower'], f'({target_limits["lower"]:.1f})', 'darkorange', 'dashdot'),
                (target_limits['upper'], f'({target_limits["upper"]:.1f})', 'darkgreen', 'dashdot'),
            ]:
                norm_val = (limit_val - pv_min) / (pv_max - pv_min)
                fig.add_hline(y=norm_val, row=1, col=1,
                              line=dict(color=limit_color, width=1.5, dash=limit_dash),
                              annotation_text=limit_name, annotation_position='right',
                              annotation_font_size=9, annotation_font_color=limit_color)

    # ── Shading ──
    n_rows = 2 if has_actions else 1

    # Cluster-level shading (light orange)
    if cluster_regions:
        for cid_r, c_start_r, c_end_r in cluster_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=c_start_r, x1=c_end_r, fillcolor='rgba(255, 165, 0, 0.10)',
                              line=dict(width=0), layer='below', row=r, col=1)
                fig.add_vline(x=c_start_r, line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)
                fig.add_vline(x=c_end_r, line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)

    # Individual alarm shading (light red)
    if alarm_regions:
        for alarm_start, alarm_end, ep_num in alarm_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=alarm_start, x1=alarm_end, fillcolor='rgba(255, 0, 0, 0.12)',
                              line=dict(color='red', width=0.5, dash='dot'), layer='below', row=r, col=1)

    # ── Bottom subplot: Control actions ──
    if has_actions:
        action_sources = sorted(actions['Source'].unique())
        source_y_map = {src: i for i, src in enumerate(action_sources)}

        for _, row in actions.iterrows():
            desc = str(row['Description'])
            direction = str(row['action_direction'])

            if desc in ('SP', 'OP'):
                color = 'red' if direction == 'decrease' else ('green' if direction == 'increase' else 'grey')
            elif desc == 'MODE':
                color = 'blue'
            else:
                color = 'grey'

            symbol = 'diamond' if desc == 'SP' else ('circle' if desc == 'OP' else ('square' if desc == 'MODE' else 'x'))
            prev_val = str(row['PrevValue'])
            curr_val = str(row['Value'])
            timing = str(row['action_timing'])

            fig.add_trace(
                go.Scatter(
                    x=[row['VT_Start']], y=[source_y_map[row['Source']]], mode='markers',
                    marker=dict(color=color, size=10, symbol=symbol,
                                line=dict(width=1, color='darkgrey')),
                    showlegend=False,
                    hovertemplate=(
                        f'<b>{row["Source"]}</b> ({desc})<br>'
                        f'Time: %{{x}}<br>Direction: {direction}<br>'
                        f'{prev_val} \u2192 {curr_val}<br>'
                        f'Timing: {timing}<br><extra></extra>'
                    ),
                ), row=2, col=1)

        fig.update_yaxes(tickvals=list(source_y_map.values()), ticktext=list(source_y_map.keys()),
                         row=2, col=1, title_text='Tag', gridcolor='rgba(200,200,200,0.3)')

        # Action color legend entries
        for label, clr, sym in [('SP Increase', 'green', 'diamond'), ('SP Decrease', 'red', 'diamond'),
                                 ('OP Increase', 'green', 'circle'), ('OP Decrease', 'red', 'circle'),
                                 ('MODE Change', 'blue', 'square'), ('Other Action', 'grey', 'x')]:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                marker=dict(color=clr, size=10, symbol=sym, line=dict(width=1, color='darkgrey')),
                name=label, legendgroup='action_legend', legendgrouptitle_text='Actions'))

    # ── Layout ──
    fig.update_layout(
        title=title, height=750 if has_actions else 550, template='plotly_white',
        hovermode='closest',
        legend=dict(groupclick='toggleitem', tracegroupgap=3, font=dict(size=10)),
        xaxis=dict(title=''))
    fig.update_yaxes(showticklabels=False, title_text='', row=1, col=1)

    return fig

In [7]:
# # Generate all cluster plots
# cluster_ids = sorted(cluster_info.keys())
# target_limits = {'lower': TARGET_PV_LOWER, 'upper': TARGET_PV_UPPER, 'alarm': TARGET_ALARM_LIMIT}
# print(f'Generating plots for {len(cluster_ids)} clusters...')

# for i, cid in enumerate(cluster_ids):
#     cdata = cluster_info[cid]
#     cactions = cluster_actions.get(cid, None)

#     c_start = cdata['cluster_start']
#     c_end = cdata['cluster_end']
#     window_start = c_start - pd.Timedelta(minutes=30)
#     window_end = c_end + pd.Timedelta(minutes=30)
#     n_alarms = cdata['total_alarms']
#     c_type = cdata['cluster_type']

#     title = (f'Cluster {cid} | {n_alarms} alarm(s) | {c_type} | '
#              f'{c_start.strftime("%Y-%m-%d %H:%M")} to {c_end.strftime("%Y-%m-%d %H:%M")}')

#     fig = create_window_plot(
#         window_start, window_end, op_pv_data_df, ordered_tags, base_colors,
#         title=title, actions=cactions,
#         cluster_regions=[(cid, c_start, c_end)],
#         alarm_regions=cdata['alarms'],
#         target_limits=target_limits)

#     if fig is not None:
#         outpath = RESULTS_DIR / f'cluster_{cid:04d}.html'
#         fig.write_html(str(outpath), include_plotlyjs='cdn')

#     if (i + 1) % 10 == 0 or (i + 1) == len(cluster_ids):
#         print(f'  {i + 1}/{len(cluster_ids)} done')

# print(f'\nAll plots saved to {RESULTS_DIR}/')

In [8]:
# ── Custom time window plot ──
# Specify start and end timestamps below
START_TIME = '2024-01-06 00:00'
END_TIME   = '2024-01-06 23:59'

start_ts = pd.Timestamp(START_TIME)
end_ts = pd.Timestamp(END_TIME)

# Find alarm clusters that overlap with this window
overlapping_alarms = []
overlapping_clusters = []
for cid, cdata in cluster_info.items():
    c_start = cdata['cluster_start']
    c_end = cdata['cluster_end']
    if c_start <= end_ts and c_end >= start_ts:
        overlapping_clusters.append((cid, c_start, c_end))
        for alarm_start, alarm_end, ep_num in cdata['alarms']:
            if alarm_start <= end_ts and alarm_end >= start_ts:
                overlapping_alarms.append((alarm_start, alarm_end, ep_num))

# Get control actions from raw events (not the clustered Excel)
window_actions = raw_change_events[
    (raw_change_events['VT_Start'] >= start_ts) & (raw_change_events['VT_Start'] <= end_ts)
].copy()

if len(window_actions) > 0:
    # Compute action_direction from Value vs PrevValue
    _val_num = pd.to_numeric(window_actions['Value'], errors='coerce')
    _prev_num = pd.to_numeric(window_actions['PrevValue'], errors='coerce')
    window_actions['action_direction'] = np.where(
        _val_num > _prev_num, 'increase',
        np.where(_val_num < _prev_num, 'decrease', 'no_change'))

    # Compute action_timing relative to overlapping clusters (if any)
    def get_timing(ts):
        for _, c_start, c_end in overlapping_clusters:
            if ts < c_start:
                return 'before'
            elif ts > c_end:
                return 'after'
            else:
                return 'during'
        return 'no_cluster'
    window_actions['action_timing'] = window_actions['VT_Start'].apply(get_timing)
else:
    window_actions = None

print(f'Window: {start_ts} to {end_ts}')
print(f'Overlapping clusters: {len(overlapping_clusters)}, Individual alarms: {len(overlapping_alarms)}')
print(f'Control actions in window (from raw events): {len(window_actions) if window_actions is not None else 0}')

target_limits = {'lower': TARGET_PV_LOWER, 'upper': TARGET_PV_UPPER, 'alarm': TARGET_ALARM_LIMIT}
fig = create_window_plot(
    start_ts, end_ts, op_pv_data_df, ordered_tags, base_colors,
    title=f'Custom Window: {START_TIME} to {END_TIME}',
    actions=window_actions,
    cluster_regions=overlapping_clusters if overlapping_clusters else None,
    alarm_regions=overlapping_alarms if overlapping_alarms else None,
    target_limits=target_limits)

if fig is not None:
    fig.show()

Window: 2024-01-06 00:00:00 to 2024-01-06 23:59:00
Overlapping clusters: 11, Individual alarms: 15
Control actions in window (from raw events): 12


In [9]:
# Check raw events for 03PIC_1013 control actions between May 5, 2025 06:00 - 12:00
raw_events = pd.read_csv(DATA_DIR / 'df_df_events_1071_export.csv', low_memory=False)
raw_events['VT_Start'] = pd.to_datetime(raw_events['VT_Start'])

mask = (
    raw_events['Source'].str.contains('1071', na=False) &
    (raw_events['VT_Start'] >= '2025-01-08 02:00') &
    (raw_events['VT_Start'] <= '2025-01-08 04:00')
)
result = raw_events.loc[mask, ['Source', 'ConditionName', 'Description', 'PrevValue', 'Value', 'VT_Start', 'Category']].copy()
result = result.sort_values('VT_Start')
print(f'Found {len(result)} events for *1013* between 2025-05-05 06:00 and 12:00:\n')
result

Found 31 events for *1013* between 2025-05-05 06:00 and 12:00:



,Source,ConditionName,Description,PrevValue,Value,VT_Start,Category
54205,03LIC_1071,PVLO,3E107 LEVEL,NaN,28.647,2025-01-08 02:51:46.114100,1
54214,03LIC_1071,PVLO,PVLO LOW 3E107 LEVEL ...,NaN,NaN,2025-01-08 02:52:19.859900,7
54218,03LIC_1071,PVLO,3E107 LEVEL,NaN,31.769,2025-01-08 02:52:20.031200,1
54226,03LIC_1071,NaN,3E107 LEVEL MODE AUTO ...,NaN,NaN,2025-01-08 03:00:40.813000,7
632273,03LIC_1071,CHANGE,MODE,AUTO,MAN,2025-01-08 03:00:40.813000,7
54223,03LIC_1071,CHANGE,MODE,NaN,MAN,2025-01-08 03:00:40.813000,7
54232,03LIC_1071,CHANGE,OP,NaN,58.0000,2025-01-08 03:00:44.933000,7
54235,03LIC_1071,NaN,3E107 LEVEL OP 52.280...,NaN,NaN,2025-01-08 03:00:44.933000,7
632277,03LIC_1071,CHANGE,OP,52.2807,58.0000,2025-01-08 03:00:44.933000,7
54244,03LIC_1071,CHANGE,OP,NaN,52.0000,2025-01-08 03:02:43.849900,7


In [10]:
# Find clusters with continuous stretches of 03LIC_1071.PV <= 0
ZERO_THRESHOLD = 0
LONG_STRETCH_MIN = 30  # minutes

clusters_agg = alarms_df.groupby('cluster_id').agg(
    cluster_start=('cluster_start_time', 'first'),
    cluster_end=('cluster_end_time', 'first')
)

long_stretches = []   # >= 30 min
short_stretches = []  # < 30 min but > 0

for cid, row in clusters_agg.iterrows():
    ws = row['cluster_start'] - pd.Timedelta(minutes=30)
    we = row['cluster_end'] + pd.Timedelta(minutes=30)
    
    window = op_pv_data_df.loc[ws:we, '03LIC_1071.PV']
    if window.empty:
        continue
    
    below_zero = (window <= ZERO_THRESHOLD).astype(int)
    if below_zero.sum() == 0:
        continue
    
    # Group consecutive runs
    groups = (below_zero != below_zero.shift()).cumsum()
    
    for _, grp in window.groupby(groups):
        if (grp <= ZERO_THRESHOLD).all() and len(grp) > 1:
            dur_min = (grp.index[-1] - grp.index[0]).total_seconds() / 60
            info = (cid, dur_min, grp.min(), grp.index[0], grp.index[-1])
            if dur_min >= LONG_STRETCH_MIN:
                long_stretches.append(info)
            elif dur_min > 0:
                short_stretches.append(info)

print(f'Total clusters checked: {len(clusters_agg)}')
print(f'\n{"="*70}')
print(f'Clusters with >= {LONG_STRETCH_MIN} min continuous PV <= {ZERO_THRESHOLD}:  '
      f'{len(set(c[0] for c in long_stretches))} clusters, {len(long_stretches)} stretches')
print(f'{"="*70}')
for cid, dur, minv, s, e in sorted(long_stretches):
    print(f'  Cluster {cid:>4d}: {dur:>6.0f} min | min PV = {minv:.2f} | {s} → {e}')

print(f'\n{"="*70}')
print(f'Clusters with < {LONG_STRETCH_MIN} min continuous PV <= {ZERO_THRESHOLD}:  '
      f'{len(set(c[0] for c in short_stretches))} clusters, {len(short_stretches)} stretches')
print(f'{"="*70}')
for cid, dur, minv, s, e in sorted(short_stretches):
    print(f'  Cluster {cid:>4d}: {dur:>6.0f} min | min PV = {minv:.2f} | {s} → {e}')

# Summary
all_affected = set(c[0] for c in long_stretches + short_stretches)
print(f'\n{"="*70}')
print(f'SUMMARY: {len(all_affected)} clusters affected out of {len(clusters_agg)} total')
print(f'  Long (>= {LONG_STRETCH_MIN} min): cluster IDs = {sorted(set(c[0] for c in long_stretches))}')
print(f'  Short (< {LONG_STRETCH_MIN} min): cluster IDs = {sorted(set(c[0] for c in short_stretches))}')

Total clusters checked: 539

Clusters with >= 30 min continuous PV <= 0:  9 clusters, 9 stretches
  Cluster   80:    100 min | min PV = -0.51 | 2022-06-21 14:29:00 → 2022-06-21 16:09:00
  Cluster  214:     45 min | min PV = -0.54 | 2023-12-26 18:09:00 → 2023-12-26 18:54:00
  Cluster  216:    152 min | min PV = -0.54 | 2023-12-27 01:52:00 → 2023-12-27 04:24:00
  Cluster  432:     34 min | min PV = -0.53 | 2024-09-24 06:28:00 → 2024-09-24 07:02:00
  Cluster  434:     34 min | min PV = -1.31 | 2024-09-26 08:27:00 → 2024-09-26 09:01:00
  Cluster  475:    160 min | min PV = -1.30 | 2025-01-08 09:19:00 → 2025-01-08 11:59:00
  Cluster  476:    257 min | min PV = -1.31 | 2025-01-08 14:50:00 → 2025-01-08 19:07:00
  Cluster  481:    101 min | min PV = -1.30 | 2025-01-12 09:51:00 → 2025-01-12 11:32:00
  Cluster  534:     72 min | min PV = -1.33 | 2025-06-21 18:29:00 → 2025-06-21 19:41:00

Clusters with < 30 min continuous PV <= 0:  13 clusters, 23 stretches
  Cluster   80:     11 min | min PV = -